In [31]:
# Libraries for preprocessing 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# Libraries for model training and performance metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [32]:
# Load dataset
data = pd.read_csv("D:/Sanctum/AI-ML-JOURNEY/Deep Learning/Data/Churn_Modelling.csv")

In [33]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [34]:
# Droping Irrelevant columns
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [35]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [36]:
# Handling gender categorical feature
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


In [37]:
# Handling Geography categorical column
ohe_geography = OneHotEncoder()
geo_ohe = ohe_geography.fit_transform(data[['Geography']]).toarray() # returns 2d array/df
geo_encoded_df = pd.DataFrame(geo_ohe,columns=ohe_geography.get_feature_names_out(['Geography'])) # columns name
# * combining with the original df
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [38]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [39]:
# Dividing the data into independent and dependent feature
X = data.drop('Exited',axis=1)
y = data['Exited']

In [40]:
# train-test split
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.20, random_state=42)
# Scale down 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
# Saving encoder to pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open("ohe_geography.pkl",'wb') as file:
    pickle.dump(ohe_geography,file)
# saving scaler to pickle
with open("scaler.pkl",'wb') as file:
    pickle.dump(scaler,file)

### ANN Implementation
* In Tensorflow it is know as sequential model/network.
* Total trainable parameter in this problem is 20 = HL1(2x3 = 6w + 3 bias) + HL2(3x2=62 + 2 bias) + OL(2x1 =2w +1 bias) example for 2 input.
* "Dense" is used for hidden neuron

In [42]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [43]:
#Building ANN model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), # HL1 connected to i/p layer
    Dense(32,activation='relu'), # HL2 connected
    Dense(1,activation='sigmoid') # OP connected
]
    
)

c:\Users\sharm\anaconda3\envs\ai-ml\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [44]:
# total no. of parmeter 
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
# configuring optimizer and loss
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

In [46]:
# Compiling the model to perform forward and backward propogation
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [47]:
# Set up the tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [48]:
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [49]:
# Set up Early stopping -> this is used to check if the loss is actually decreasing. Set up a threshold epoch value to stop in case of no improvement
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [50]:
# Training model
history = model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8350 - loss: 0.3980 - val_accuracy: 0.8560 - val_loss: 0.3587
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8562 - loss: 0.3529 - val_accuracy: 0.8585 - val_loss: 0.3595
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8564 - loss: 0.3473 - val_accuracy: 0.8620 - val_loss: 0.3453
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8604 - loss: 0.3432 - val_accuracy: 0.8580 - val_loss: 0.3541
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8593 - loss: 0.3421 - val_accuracy: 0.8580 - val_loss: 0.3453
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8610 - loss: 0.3381 - val_accuracy: 0.8595 - val_loss: 0.3467
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8622 - loss: 0.3367 - val_accuracy: 0.8620 - val_loss: 0.3442
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8608 - loss: 0.3330 - val_accu

In [51]:
# Saving model
model.save('model.h5') # h5 is compatible with keras

In [52]:
# lauch tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [53]:
# !pip install tensorboard notebook jupyterlab ## needed for my incompatible environment and ui not rendered

In [54]:
%tensorboard --logdir logs/fit # bacause of + adding the date time as string

Reusing TensorBoard on port 6006 (pid 33884), started 0:11:30 ago. (Use '!kill 33884' to kill it.)

In [ ]:
# loading the pickle file